In [2]:
import pandas as pd
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import nltk
import re
import numpy as np



# Load data
df = pd.read_csv("/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv")  # replace with your actual path
df.drop(columns=['Unnamed: 0','date', 'usefulCount', 'review_length'], inplace=True)
# Drop rows with missing reviews or ratings
df = df.dropna(subset=['review', 'rating'])
df.drop(['patient_id', 'condition'], axis=1, inplace=True)

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

def clean_text(text):
    # Lowercase, remove punctuation and digits
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text
df['review_clean'] = df['review'].astype(str).apply(clean_text)
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['review_clean'])



In [7]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X_bow, df['rating'], test_size=0.2, random_state=42)

lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

MSE: 9.393897268365066


Train & Test from Clean Part

In [14]:


train_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv')
test_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv')

train_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)
test_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)

vectorizer = CountVectorizer(max_features=60000)  # Limit features if needed
X_train_bow = vectorizer.fit_transform(train_df['review_clean'])
X_test_bow = vectorizer.transform(test_df['review_clean'])

y_train = train_df['rating'].values
y_test = test_df['rating'].values

vectorizer = CountVectorizer(max_features=60000)  # Limit features if needed
X_train_bow = vectorizer.fit_transform(train_df['review_clean'])
X_test_bow = vectorizer.transform(test_df['review_clean'])

y_train = train_df['rating'].values
y_test = test_df['rating'].values


lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train_bow, y_train)

y_pred = lasso.predict(X_test_bow)
print("Test MSE:", mean_squared_error(y_test, y_pred))

Test MSE: 9.806323542452168


In [15]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

# === 1. Load data ===
train_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv')
test_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv')

# Drop unneeded columns
train_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)
test_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)

# === 2. Feature Extraction (BoW ) ===
vectorizer = CountVectorizer(max_features=60000)
X_train_bow = vectorizer.fit_transform(train_df['review_clean'])
X_test_bow = vectorizer.transform(test_df['review_clean'])

y_train = train_df['rating'].values
y_test = test_df['rating'].values
vocab = vectorizer.get_feature_names_out()

In [16]:
# === LASSO for Sparse Interpretability ===
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train_bow, y_train)
y_pred_lasso = lasso.predict(X_test_bow)

print("Test MSE with LASSO:", mean_squared_error(y_test, y_pred_lasso))

# Show top non-zero weighted features
nonzero_indices = np.where(lasso.coef_ != 0)[0]
top_features = sorted(zip(nonzero_indices, lasso.coef_[nonzero_indices]), key=lambda x: -abs(x[1]))[:15]
print("\nTop LASSO features:")
for idx, weight in top_features:
    print(f"{vocab[idx]}: {weight:.3f}")


Test MSE with LASSO: 9.806323542452168

Top LASSO features:
bad: -0.496
year: 0.347
stop: -0.335
life: 0.288
work: 0.259
horrible: -0.177
good: 0.167
love: 0.158
bleed: -0.109
great: 0.058
effect: 0.049
mg: 0.037
month: -0.031
take: -0.021


In [17]:
# === Dimensionality Reduction via SVD (PCA for sparse data) ===
n_components = 50
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_svd = svd.fit_transform(X_train_bow)
X_test_svd = svd.transform(X_test_bow)

# === Linear Regression on PCA features ===
reg = LinearRegression()
reg.fit(X_train_svd, y_train)
y_pred_pca = reg.predict(X_test_svd)

print("Test MSE with PCA + Linear Regression:", mean_squared_error(y_test, y_pred_pca))


Test MSE with PCA + Linear Regression: 8.915361152666415


In [18]:
# === Inspect Top Words in PCA Dimensions ===
def print_top_words_per_component(svd, vocab, n_words=10, n_components=5):
    print("\nTop words per PCA component:")
    for i in range(n_components):
        comp = svd.components_[i]
        top_indices = np.argsort(np.abs(comp))[-n_words:][::-1]
        words = [vocab[j] for j in top_indices]
        print(f"Component {i + 1}: {', '.join(words)}")

print_top_words_per_component(svd, vocab)



Top words per PCA component:
Component 1: day, take, feel, month, year, start, week, work, go, time
Component 2: period, month, mg, pill, day, feel, get, birth, control, take
Component 3: day, year, period, month, work, effect, mg, anxiety, pill, try
Component 4: pain, feel, year, like, week, take, work, start, anxiety, doctor
Component 5: take, pain, feel, pill, like, get, go, year, bad, start


In [19]:
nonzero_idx = np.where(lasso.coef_ != 0)[0]
nonzero_weights = lasso.coef_[nonzero_idx]
nonzero_words = vectorizer.get_feature_names_out()[nonzero_idx]

top_positive = sorted(zip(nonzero_words, nonzero_weights), key=lambda x: -x[1])[:10]
top_negative = sorted(zip(nonzero_words, nonzero_weights), key=lambda x: x[1])[:10]

print("\nTop Positive Influential Words:")
for word, weight in top_positive:
    print(f"{word}: {weight:.3f}")

print("\nTop Negative Influential Words:")
for word, weight in top_negative:
    print(f"{word}: {weight:.3f}")



Top Positive Influential Words:
year: 0.347
life: 0.288
work: 0.259
good: 0.167
love: 0.158
great: 0.058
effect: 0.049
mg: 0.037
take: -0.021
month: -0.031

Top Negative Influential Words:
bad: -0.496
stop: -0.335
horrible: -0.177
bleed: -0.109
month: -0.031
take: -0.021
mg: 0.037
effect: 0.049
great: 0.058
love: 0.158


In [20]:
component_weights = reg.coef_
for i, weight in enumerate(component_weights[:10]):
    print(f"Component {i+1}: Weight = {weight:.3f}")


Component 1: Weight = 0.130
Component 2: Weight = -0.239
Component 3: Weight = -0.257
Component 4: Weight = 0.141
Component 5: Weight = -0.107
Component 6: Weight = -0.582
Component 7: Weight = -0.412
Component 8: Weight = -0.223
Component 9: Weight = -0.001
Component 10: Weight = 0.393
